# Flujo completo L4: face swap + realismo Z-Image (bf16) + puntuación, reintentos adaptativos y textura «candid»
Runtime: **GPU L4** (24 GB, bf16 nativo). Si al inicio detecta otra GPU o <20 GB, avisa y usa el **modo T4** (swaps primero + fp16→bf16). Sin Drive: todo en disco local de la sesión (`/content`).

**Por cada destino:** swap (FaceFusion 3.9.1, `hyperswap_1a_256`, pixel-boost 512, `gpen_bfr_512`) → img2img de baja fuerza con Z-Image Turbo sobre el recorte de la cara (+ LoRA, paste-back, mezcla solo-textura, grano) → **puntuación** (ArcFace vs origen, cara detectada, cambio de fondo) → si no pasa el umbral, **reintenta variando semilla / strength / blend** (hasta `MAX_INTENTOS`) → guarda el mejor intento → **posproceso candid** opcional (highlights, color, nitidez, grano, ruido HF, viñeta) → tabla de puntuaciones, miniaturas y zip.

En L4 Z-Image y FaceFusion conviven en VRAM: cada intento hace swap → img2img → puntuación, y el siguiente intento ajusta strength/blend según esa puntuación.

Orden de celdas: instalación + GPU → parámetros → imágenes → ArcFace → swap → utilidades → candid → Z-Image → puntuación → bucle → reporte → zip.

In [ ]:
# 1) Instalación. FaceFusion va en un venv aislado (/content/ffvenv); diffusers/insightface en el entorno de Colab.
#    La descarga de Z-Image arranca en segundo plano mientras se instala FaceFusion.
import subprocess, time, os, sys, glob, shutil, gc, json, threading
def sh(cmd, timeout=1800, show=True, env=None):
    t = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash", timeout=timeout, env=env)
    out = r.stdout + r.stderr
    if show: print(f"$ {cmd[:120]}  [{time.time()-t:.0f}s rc={r.returncode}]\n{out[-700:]}")
    return r.returncode, out
T0 = time.time()
sh("pip uninstall -y -q torchao")   # el torchao viejo de Colab rompe diffusers
sh("pip install -q -U diffusers transformers accelerate safetensors sentencepiece peft huggingface_hub psutil")
sh("pip install -q insightface onnxruntime")   # detector + ArcFace (CPU basta)
def prefetch():                                 # modelo base de Z-Image en la caché local de HF
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("Tongyi-MAI/Z-Image-Turbo"); print("[prefetch] Z-Image Turbo descargado")
    except Exception as e: print("[prefetch] falló (se reintentará al cargar):", type(e).__name__, str(e)[:150])
TH = threading.Thread(target=prefetch, daemon=True); TH.start()
FF_DIR, FF_PY = "/content/facefusion", "/content/ffvenv/bin/python"
if not os.path.exists(FF_DIR): sh(f"git clone -q --depth 1 --branch 3.9.1 https://github.com/facefusion/facefusion {FF_DIR}")
sh("pip install -q uv"); sh("uv venv /content/ffvenv --python 3.13 --allow-existing")
sh(f"grep -v '^onnxruntime\\|^gradio' {FF_DIR}/requirements.txt > /tmp/ff_req.txt && uv pip install -p {FF_PY} -r /tmp/ff_req.txt onnxruntime-gpu==1.30.0")
libdirs = sorted(set(glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib") + glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cu13/lib")
                     + ["/usr/local/cuda/lib64", "/usr/lib/x86_64-linux-gnu"]))
FF_ENV = dict(os.environ, LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
sh(f'{FF_PY} -c "import onnxruntime as o, numpy; print(o.__version__, numpy.__version__, o.get_available_providers())"', env=FF_ENV)
import torch, diffusers
from diffusers import ZImageImg2ImgPipeline
# Verificación de GPU: L4 (o cualquiera con ≥20 GB) → modo L4; si no, aviso y modo T4 del cuaderno original
GPU_NOMBRE = torch.cuda.get_device_name(0); GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
MODO = "L4" if ("L4" in GPU_NOMBRE and GPU_GB >= 20) else "T4"
if MODO == "T4": print(f"⚠️ GPU {GPU_NOMBRE} ({GPU_GB:.0f} GB) no es L4 con ≥20 GB → MODO T4: swaps primero, Z-Image fp16 con fallback bf16, lado 768")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True   # barato en Ampere/Ada (L4); en T4 no hace nada
print("diffusers", diffusers.__version__, "| torch", torch.__version__, f"| GPU {GPU_NOMBRE} {GPU_GB:.0f} GB | MODO {MODO}")
T_INSTALL = time.time() - T0; print(f"Instalación: {T_INSTALL:.0f}s")

In [ ]:
# 2) PARÁMETROS (edita aquí) ------------------------------------------------------------
ORIGEN_DIR  = "/content/origen"      # 1 imagen con la cara de origen (si está vacía, se pide subirla)
DESTINOS_DIR = "/content/destinos"   # 1 o varias imágenes destino (si está vacía, se pide subirlas)
OUT_DIR     = "/content/resultados"

UMBRAL_ARCFACE = 0.45   # coseno mínimo (resultado vs ORIGEN, buffalo_l). Un swap bueno suele dar 0.4–0.7; sube para ser más exigente
FONDO_MAX      = 0.03   # cambio medio máximo (0-1) del fondo (fuera de la cara) respecto al swap; el grano suma ≈0.008
MAX_INTENTOS   = 4      # intentos por destino (se detiene en el primero que pasa)
# Modo L4 (adaptativo): tras cada intento se ajusta según la puntuación:
#  - ArcFace cae respecto al swap o el fondo excede → baja strength un PASO_STRENGTH (Z-Image reescribió demasiado)
#  - el propio swap ya queda bajo el umbral → baja el blend de GPEN un PASO_BLEND (más cara del swap). Si el swap pasa, blend se queda en 80
#    (bajar el blend empeoró la piel en pruebas). Modo T4: programa lineal fijo del cuaderno original.
PASO_STRENGTH, PASO_BLEND = 0.03, 10

# Variación entre intentos: el intento 1 usa los valores iniciales; los siguientes se desplazan linealmente hacia el extremo "más identidad"
# (menos strength = menos reescritura; menos blend del enhancer = más cara del swap) y cambian la semilla.
SEMILLA_BASE   = 0
SEMILLA_PASO   = 17
STRENGTH_INI, STRENGTH_RANGO = 0.20, (0.15, 0.30)
BLEND_INI,    BLEND_RANGO    = 80, (50, 100)

# Swap (FaceFusion): fijo como en la versión probada
FF_ARGS = ("--processors face_swapper face_enhancer --face-swapper-model hyperswap_1a_256 --face-swapper-pixel-boost 512x512 "
           "--face-enhancer-model gpen_bfr_512 --face-selector-mode one --face-selector-order large-small "
           "--execution-providers cuda --output-image-quality 100 --log-level info")

# Z-Image Turbo img2img (sobre el recorte de la cara)
A_PROMPT = "candid unretouched smartphone photo, natural skin texture with visible pores, realistic skin, no smoothing"
A_STEPS, A_GUIDANCE = 8, 0.0
A_SIDE_L4, A_SIDE_T4 = 1024, 768                # lado de proceso del recorte: 1024 en L4 (cabe holgado en bf16), 768 en T4
CROP_MARGEN, FEATHER = 2.0, 0.6                   # margen del recorte, degradado del paste-back
MEZCLA, TEX_SIGMA, TEX_GAIN, OPACIDAD = "textura", 2.0, 1.0, 0.9   # mezcla solo-textura (high-pass) para no cambiar la identidad
CARAS_A_EDITAR = 1                                # caras más grandes por imagen que pasan por Z-Image (la 1.ª es la puntuada)
GRANO = 0.010                                     # grano por intento (0 = sin grano). Con APLICAR_CANDID se omite: el grano lo pone candid
# LoRA: se intenta la pedida (ang187, requiere aceptar acceso y HF_TOKEN en secretos de Colab); si no se puede, sigue con la pública de respaldo, luego sin LoRA
A_LORA = ("ang187/realstagram-zimg", "REALSTAGRAM_ZIMG.safetensors")
A_LORA_RESPALDO = ("JosephMinchala/JosephMinchala-skin-texture-zturbo-v45", "skin texture Photorealistic style v4.5.safetensors")
A_LORA_W = 0.3

# Corrección por zonas (ojos con landmarks): STUB opcional, desactivado. Ver celda 8.
CORREGIR_OJOS = False

# Textura «candid» (posproceso sin modelo del cuaderno face_swap_candid_lotes), aplicada al MEJOR intento tras puntuar
APLICAR_CANDID = True
PP = dict(
    highlights=dict(on=True, knee=0.66, strength=0.35, skin_only=True),   # compresión de brillos (solo piel/cara)
    color=dict(on=True, contrast=1.04, saturation=0.98, warmth=0.004, fade_blacks=0.012),
    sharpen=dict(on=True, amount=0.30, radius=1.1, threshold=0.012),       # unsharp suave
    grain=dict(on=True, intensity=0.017, size=1.4, midtone_bias=0.7, chroma=0.05),
    hf_noise=dict(on=True, intensity=0.006, skin_boost=1.5),               # ruido fino de sensor, más en piel
    vignette=dict(on=True, strength=0.15, power=2.2),
    jpeg=dict(on=False, quality=92),
    ref_long=1500, seed=1234)

In [ ]:
# 3) Imágenes: si las carpetas ya tienen imágenes se salta la subida. Todo se normaliza a PNG (FaceFusion exige misma extensión de salida).
import cv2, numpy as np
from PIL import Image, ImageOps
EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
list_imgs = lambda d: sorted(f for f in os.listdir(d) if f.lower().endswith(EXTS)) if os.path.isdir(d) else []
for d in (ORIGEN_DIR, DESTINOS_DIR): os.makedirs(d, exist_ok=True)
if not list_imgs(ORIGEN_DIR) or not list_imgs(DESTINOS_DIR):
    from google.colab import files
    if not list_imgs(ORIGEN_DIR):
        print("Sube la imagen ORIGEN (la cara):"); up = files.upload()
        for n in up: shutil.move(n, os.path.join(ORIGEN_DIR, n))
    if not list_imgs(DESTINOS_DIR):
        print("Sube las imágenes DESTINO:"); up = files.upload()
        for n in up: shutil.move(n, os.path.join(DESTINOS_DIR, n))
PNG_SRC, PNG_DST = "/content/_origen_png", "/content/_destinos_png"
for d in (PNG_SRC, PNG_DST): shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
def a_png(ruta, out_dir):
    img = cv2.imread(ruta)                       # aplica EXIF
    if img is None: return None
    q = os.path.join(out_dir, os.path.splitext(os.path.basename(ruta))[0] + ".png"); cv2.imwrite(q, img); return q
SRC_PNG = a_png(os.path.join(ORIGEN_DIR, list_imgs(ORIGEN_DIR)[0]), PNG_SRC)
DESTINOS = [(f, a_png(os.path.join(DESTINOS_DIR, f), PNG_DST)) for f in list_imgs(DESTINOS_DIR)]
DESTINOS = [(f, q) for f, q in DESTINOS if q]
print("Origen:", SRC_PNG, "| Destinos:", [f for f, _ in DESTINOS])

In [ ]:
# 4) Detección + ArcFace (insightface buffalo_l: detector y reconocimiento). Se calcula el embedding del ORIGEN una vez.
from insightface.app import FaceAnalysis
FA = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"], allowed_modules=["detection", "recognition"])
FA.prepare(ctx_id=-1, det_size=(640, 640))
area = lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1])
def caras(bgr): return sorted(FA.get(bgr), key=lambda f: -area(f))     # de mayor a menor
f_src = caras(cv2.imread(SRC_PNG)); assert f_src, "No se detectó cara en el ORIGEN"
EMB_SRC = f_src[0].normed_embedding
VALIDOS = []
for f, q in DESTINOS:
    n = len(caras(cv2.imread(q)))
    if n: VALIDOS.append((f, q)); print(f"✅ {f}: {n} cara(s)")
    else: print(f"⚠️ {f}: sin cara detectada, se omite")
assert VALIDOS, "No hay destinos con cara"

In [ ]:
# 5) Swap con FaceFusion (headless-run con proveedor CUDA, un proceso por intento; caché LOCAL por destino+blend)
SWAP_DIR = "/content/_swap_cache"; os.makedirs(SWAP_DIR, exist_ok=True)
def swap(dst_png, blend):
    out = os.path.join(SWAP_DIR, f"{os.path.splitext(os.path.basename(dst_png))[0]}_b{blend}.png")
    if os.path.exists(out): return out
    rc, log = sh(f"cd {FF_DIR} && {FF_PY} facefusion.py headless-run -s {SRC_PNG} -t {dst_png} -o {out} --jobs-path /content/_jobs "
                 f"{FF_ARGS} --face-enhancer-blend {blend}", env=FF_ENV, timeout=1500, show=False)
    if rc != 0 or not os.path.exists(out): raise RuntimeError("FaceFusion falló:\n" + log[-1500:])
    return out

In [ ]:
# 5b) Parámetros por intento. MODO T4: programa lineal fijo y swaps por adelantado (Z-Image ocupa ~12 GB de la T4).
#     MODO L4: hay VRAM para ambos, así que el swap se hace dentro del bucle y siguiente_intento() ajusta con la puntuación previa.
def params_intento(k):
    t = k / max(MAX_INTENTOS - 1, 1)                       # 0 → valores iniciales; 1 → extremo «más identidad»
    return dict(semilla=SEMILLA_BASE + SEMILLA_PASO * k,
                strength=round(STRENGTH_INI + (STRENGTH_RANGO[0] - STRENGTH_INI) * t, 3),
                blend=int(round(BLEND_INI + (BLEND_RANGO[0] - BLEND_INI) * t)))
def siguiente_intento(k, prev, S):
    if MODO == "T4" or prev is None or S.get("sim") is None: return params_intento(k)
    P = dict(prev, semilla=SEMILLA_BASE + SEMILLA_PASO * k)
    if S["sim_swap"] is not None and S["sim_swap"] < UMBRAL_ARCFACE:                  # el swap mismo no llega: más cara del swap
        P["blend"] = max(BLEND_RANGO[0], P["blend"] - PASO_BLEND)
    if (S["sim_swap"] is not None and S["sim"] < S["sim_swap"] - 0.02) or (S["fondo"] or 0) > FONDO_MAX:   # Z-Image se pasó
        P["strength"] = round(max(STRENGTH_RANGO[0], P["strength"] - PASO_STRENGTH), 3)
    return P
if MODO == "T4":
    for f, q in VALIDOS:
        for k in range(MAX_INTENTOS):
            t = time.time(); swap(q, params_intento(k)["blend"]); print(f"swap {f} blend {params_intento(k)['blend']}: {time.time()-t:.0f}s")
else: print("MODO L4: swaps intercalados en el bucle")

In [ ]:
# 6) Utilidades Z-Image: recorte con margen, máscara suave, mezcla solo-textura (high-pass) y grano
def recorte_cuadrado(shape, box, margen):
    h, w = shape[:2]; cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
    s = max(int(min(max(box[2]-box[0], box[3]-box[1]) * margen, h, w)), 64)
    return int(np.clip(cx - s/2, 0, w - s)), int(np.clip(cy - s/2, 0, h - s)), s
def mascara_suave(s, feather):
    yy, xx = np.mgrid[0:s, 0:s].astype(np.float32); d = np.sqrt(((xx-(s-1)/2)/(s/2))**2 + ((yy-(s-1)/2)/(s/2))**2)
    t = np.clip((1 - d) / max(feather, 1e-3), 0, 1); return t*t*(3 - 2*t)
def mezclar(orig_c, res_c, mask):
    s = orig_c.shape[0]
    if MEZCLA == "textura":
        hp = res_c - cv2.GaussianBlur(res_c, (0, 0), max(0.6, TEX_SIGMA * s / 1024))   # solo detalle fino (poros)
        cand = np.clip(orig_c + TEX_GAIN * hp, 0, 1)
    else: cand = res_c
    a = (mask * OPACIDAD)[..., None]; return orig_c * (1 - a) + cand * a
def a_pil(b): return Image.fromarray(cv2.cvtColor((np.clip(b, 0, 1)*255 + .5).astype(np.uint8), cv2.COLOR_BGR2RGB))
def de_pil(im): return cv2.cvtColor(np.asarray(im.convert("RGB")), cv2.COLOR_RGB2BGR).astype(np.float32) / 255
def salida_valida(im): a = np.asarray(im, dtype=np.float32); return np.isfinite(a).all() and a.mean() > 3 and a.std() > 3   # negro/NaN = fallo típico de fp16
def grano(img_u8, intensidad, seed):
    """Grano luma ponderado a medios tonos + algo de croma y ruido fino (misma receta del posproceso candid)."""
    if intensidad <= 0: return img_u8
    h, w = img_u8.shape[:2]; rng = np.random.default_rng(seed); im = img_u8.astype(np.float32) / 255; sc = max(h, w) / 1500
    def nz(sig):
        n = rng.standard_normal((h, w)).astype(np.float32)
        if sig > 0.45: n = cv2.GaussianBlur(n, (0, 0), sig)
        return n / (n.std() + 1e-8)
    y = 0.114*im[..., 0] + 0.587*im[..., 1] + 0.299*im[..., 2]; wgt = 0.3 + 0.7 * np.clip(4*y*(1-y), 0, 1)
    im = im + (nz(1.4*sc*0.5) * intensidad * wgt)[..., None]
    im = im + np.stack([nz(1.4*sc*0.5) for _ in range(3)], -1) * (intensidad * 0.05 * wgt[..., None])
    im = im + (nz(0) * intensidad * 0.35)[..., None]
    return (np.clip(im, 0, 1) * 255 + .5).astype(np.uint8)

In [ ]:
# 6b) Posproceso «candid» trasladado de build_candid.py (sin cambios de lógica): highlights → color/contraste → nitidez → grano → ruido HF → viñeta.
#     Las cajas de cara vienen de insightface (en candid venían del detector de FaceFusion); la máscara de piel = elipse ∩ color YCrCb.
def _luma(im): return 0.114*im[..., 0] + 0.587*im[..., 1] + 0.299*im[..., 2]
def skin_mask(img_u8, boxes):
    h, w = img_u8.shape[:2]; ycc = cv2.cvtColor(img_u8, cv2.COLOR_BGR2YCrCb)
    col = cv2.inRange(ycc, (0, 128, 70), (255, 185, 140))
    col = cv2.morphologyEx(col, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    col = cv2.morphologyEx(col, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    if boxes:
        ell = np.zeros((h, w), np.uint8)
        for x1, y1, x2, y2 in boxes: cv2.ellipse(ell, (int((x1+x2)/2), int((y1+y2)/2)), (int((x2-x1)*0.62), int((y2-y1)*0.75)), 0, 0, 360, 255, -1)
        col = cv2.bitwise_and(col, ell); k = max(3, int(np.mean([(b[2]-b[0]) for b in boxes]) * 0.10)) | 1
    else: k = max(3, int(min(h, w) * 0.02)) | 1
    return cv2.GaussianBlur(col.astype(np.float32) / 255, (k, k), 0)
def _noise(h, w, sigma_px, rng):
    n = rng.standard_normal((h, w)).astype(np.float32)
    if sigma_px > 0.45: n = cv2.GaussianBlur(n, (0, 0), sigma_px)
    return n / (n.std() + 1e-8)
def candid(img_u8, boxes, pp=None):
    pp = pp or PP; h, w = img_u8.shape[:2]; rng = np.random.default_rng(pp["seed"])
    im = img_u8.astype(np.float32) / 255; scale = max(h, w) / pp["ref_long"]
    need = (pp["highlights"]["on"] and pp["highlights"]["skin_only"]) or (pp["hf_noise"]["on"] and pp["hf_noise"]["skin_boost"] != 1)
    mask = skin_mask(img_u8, boxes) if need else None
    c = pp["highlights"]                                   # rodilla suave sobre la luma (solo baja), opcional solo piel
    if c["on"]:
        y = _luma(im); k, s = c["knee"], c["strength"]; over = np.clip(y - k, 0, None) / max(1e-3, 1 - k)
        y2 = np.minimum(np.where(y > k, k + (1 - k) * over / (1 + s * 3.0 * over), y), y); gain = (y2 + 1e-4) / (y + 1e-4)
        if c["skin_only"] and mask is not None: gain = 1 + mask * (gain - 1)
        im = np.clip(im * gain[..., None], 0, 1)
    c = pp["color"]                                        # saturación, curva S, calidez, negros levantados
    if c["on"]:
        y = _luma(im)[..., None]; im = np.clip(y + (im - y) * c["saturation"], 0, 1)
        if c["contrast"] != 1: sm = im * im * (3 - 2 * im); im = np.clip(im + (c["contrast"] - 1) * 2 * (sm - im), 0, 1)
        im[..., 2] = np.clip(im[..., 2] + c["warmth"], 0, 1); im[..., 0] = np.clip(im[..., 0] - c["warmth"], 0, 1)
        im = c["fade_blacks"] + im * (1 - c["fade_blacks"])
    c = pp["sharpen"]                                      # unsharp con umbral
    if c["on"]:
        d = im - cv2.GaussianBlur(im, (0, 0), max(0.3, c["radius"] * scale)); d = np.where(np.abs(d) < c["threshold"], 0, d)
        im = np.clip(im + c["amount"] * d, 0, 1)
    c = pp["grain"]                                        # grano de película a medios tonos + croma
    if c["on"]:
        y = _luma(im); wgt = (1 - c["midtone_bias"]) + c["midtone_bias"] * np.clip(4 * y * (1 - y), 0, 1); sg = c["size"] * scale * 0.5
        im = im + (_noise(h, w, sg, rng) * c["intensity"] * wgt)[..., None]
        if c["chroma"] > 0: im = im + np.stack([_noise(h, w, sg, rng) for _ in range(3)], -1) * (c["intensity"] * c["chroma"] * wgt[..., None])
    c = pp["hf_noise"]                                     # ruido fino de sensor, más en piel
    if c["on"]: im = im + (_noise(h, w, 0, rng) * c["intensity"] * (1 + (c["skin_boost"] - 1) * (mask if mask is not None else 0)))[..., None]
    c = pp["vignette"]
    if c["on"]:
        yy, xx = np.mgrid[0:h, 0:w].astype(np.float32); d = (((xx - w/2) / (w/2)) ** 2 + ((yy - h/2) / (h/2)) ** 2) / 2
        im = im * (1 - c["strength"] * np.clip(d, 0, 1) ** (c["power"] / 2))[..., None]
    return (np.clip(im, 0, 1) * 255 + 0.5).astype(np.uint8)
def guardar(img_u8, path_noext):
    if PP["jpeg"]["on"]: p = path_noext + ".jpg"; cv2.imwrite(p, img_u8, [cv2.IMWRITE_JPEG_QUALITY, int(PP["jpeg"]["quality"])])
    else: p = path_noext + ".png"; cv2.imwrite(p, img_u8)
    return p

In [ ]:
# 7) Z-Image Turbo img2img: el prompt se codifica una vez (text encoder solo, luego se libera) y el transformer va a la GPU: L4 → bf16 directo; T4 → fp16 con recarga bf16 si sale negro/NaN
import psutil
from huggingface_hub import hf_hub_download
TH.join()                                   # espera a la descarga en segundo plano
TDT = {"fp16": torch.float16, "bf16": torch.bfloat16}
DT_INI = "bf16" if MODO == "L4" else "fp16"
A_SIDE = A_SIDE_L4 if MODO == "L4" else A_SIDE_T4
PIPE, EMB, DT_ACT, LORA_USADO = None, None, DT_INI, None
def liberar():
    global PIPE
    PIPE = None; gc.collect(); torch.cuda.empty_cache()
def cargar_A(dtype=DT_INI):
    global PIPE, EMB, DT_ACT, LORA_USADO
    liberar(); REPO = "Tongyi-MAI/Z-Image-Turbo"
    if EMB is None:
        from diffusers import ZImagePipeline
        te = ZImagePipeline.from_pretrained(REPO, transformer=None, vae=None, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
        te.text_encoder.to("cuda")
        with torch.inference_mode(): pe, _ = te.encode_prompt(A_PROMPT, device="cuda", do_classifier_free_guidance=False)
        EMB = [p.float().cpu() for p in pe]; assert all(torch.isfinite(p).all() for p in EMB)
        del te, pe; gc.collect(); torch.cuda.empty_cache(); print("prompt codificado")
    from diffusers import ZImageTransformer2DModel
    tr = ZImageTransformer2DModel.from_pretrained(REPO, subfolder="transformer", torch_dtype=TDT[dtype], low_cpu_mem_usage=True, device_map="cuda")
    p = ZImageImg2ImgPipeline.from_pretrained(REPO, transformer=tr, text_encoder=None, tokenizer=None, torch_dtype=TDT[dtype], low_cpu_mem_usage=True).to("cuda")
    token = None
    try:
        from google.colab import userdata; token = userdata.get("HF_TOKEN")   # opcional: necesario para el LoRA con acceso restringido
    except Exception: pass
    for lora in (A_LORA, A_LORA_RESPALDO, None):                              # pedida → respaldo pública → sin LoRA
        if lora is None: print("⚠️ Sin LoRA"); break
        try:
            f = hf_hub_download(lora[0], lora[1], token=token)
            p.load_lora_weights(os.path.dirname(f), weight_name=os.path.basename(f), adapter_name="skin"); p.set_adapters(["skin"], [A_LORA_W])
            print(f"LoRA cargado: {lora[0]} peso {A_LORA_W}"); LORA_USADO = lora[0]; break
        except Exception as e: print(f"⚠️ LoRA {lora[0]} no disponible ({type(e).__name__}: {str(e)[:90]})")
    if MODO == "T4": p.vae.enable_tiling(); p.vae.tile_sample_min_size = 512; p.vae.tile_latent_min_size = 64; p.vae.tile_overlap_factor = 0.25   # sin tiling el VAE da OOM en T4
    # sin cpu offload en ningún modo: todo residente en GPU
    p.set_progress_bar_config(disable=True); PIPE = p; DT_ACT = dtype
    print(f"Z-Image cargado ({dtype}) · VRAM {torch.cuda.memory_allocated()/2**30:.1f} GB · RAM {psutil.virtual_memory().used/2**30:.1f} GB")
def gen_A(im, seed, strength):
    g = torch.Generator("cuda").manual_seed(seed)
    with torch.inference_mode():
        return PIPE(prompt=None, prompt_embeds=[e.to("cuda", TDT[DT_ACT]) for e in EMB], image=im, strength=strength, height=im.height, width=im.width,
                    num_inference_steps=A_STEPS, guidance_scale=A_GUIDANCE, generator=g).images[0]
def gen_valido(im, seed, strength):
    r = gen_A(im, seed, strength)
    if not salida_valida(r) and DT_ACT == "fp16": print("  fp16 inválido → recargando en bf16"); cargar_A("bf16"); r = gen_A(im, seed, strength)
    if not salida_valida(r): raise RuntimeError("salida negra/NaN")
    return r
cargar_A()

In [ ]:
# 8) Realismo sobre el swap (recorte de cara → Z-Image → paste-back suave → grano) y puntuación del intento
def corregir_ojos(img_u8, caras_ins):
    """STUB opcional (CORREGIR_OJOS=False): corrección por zonas de los ojos con landmarks.
    Idea: cargar buffalo_l con 'landmark_2d_106' (allowed_modules), tomar los puntos de cada ojo, crear máscara y re-mezclar esa zona
    (p. ej. desde el origen o con otra pasada img2img). Hoy no hace nada."""
    return img_u8

def realismo(swap_bgr, seed, strength):
    """→ (resultado uint8, máscara 0-1 de lo editado)."""
    out = swap_bgr.astype(np.float32) / 255; zona = np.zeros(swap_bgr.shape[:2], np.float32)
    for c in caras(swap_bgr)[:CARAS_A_EDITAR]:
        x0, y0, s = recorte_cuadrado(swap_bgr.shape, c.bbox, CROP_MARGEN); crop = out[y0:y0+s, x0:x0+s]; lado = A_SIDE
        for _ in range(3):                                           # OOM → reintenta al 75 % del lado
            sc = lado / s; g = max(16, int(s*sc)//16*16)
            try: res = de_pil(gen_valido(a_pil(cv2.resize(crop, (g, g), interpolation=cv2.INTER_AREA if sc < 1 else cv2.INTER_CUBIC)), seed, strength)); break
            except torch.cuda.OutOfMemoryError: gc.collect(); torch.cuda.empty_cache(); lado = int(lado*.75)//16*16
        res = cv2.resize(res, (s, s), interpolation=cv2.INTER_AREA if sc > 1 else cv2.INTER_CUBIC); m = mascara_suave(s, FEATHER)
        out[y0:y0+s, x0:x0+s] = mezclar(crop, res, m); zona[y0:y0+s, x0:x0+s] = np.maximum(zona[y0:y0+s, x0:x0+s], m)
    r = (np.clip(out, 0, 1)*255 + .5).astype(np.uint8)
    if CORREGIR_OJOS: r = corregir_ojos(r, caras(swap_bgr))
    return grano(r, 0 if APLICAR_CANDID else GRANO, seed), zona

def puntuar(final, swap_bgr, zona):
    """Similitud ArcFace vs ORIGEN de la cara principal, cara detectada y cambio medio del fondo (fuera de lo editado) vs el swap."""
    ref = caras(swap_bgr)[0]; cx, cy = (ref.bbox[0]+ref.bbox[2])/2, (ref.bbox[1]+ref.bbox[3])/2
    c = caras(final); det = bool(c)
    sim = -1.0
    if det:
        f = min(c, key=lambda f: ((f.bbox[0]+f.bbox[2])/2 - cx)**2 + ((f.bbox[1]+f.bbox[3])/2 - cy)**2); sim = float(np.dot(f.normed_embedding, EMB_SRC))
    fuera = zona < 0.02
    fondo = float(np.abs(final.astype(np.float32) - swap_bgr.astype(np.float32))[fuera].mean() / 255) if fuera.any() else 0.0
    sim_swap = float(np.dot(ref.normed_embedding, EMB_SRC))
    pasa = det and sim >= UMBRAL_ARCFACE and fondo <= FONDO_MAX
    score = sim - 2 * max(0, fondo - FONDO_MAX) if det else -1.0
    return dict(sim_swap=round(sim_swap, 3), sim=round(sim, 3), cara=det, fondo=round(fondo, 4), score=round(score, 3), pasa=pasa)

In [ ]:
# 9) Bucle principal: por destino, hasta MAX_INTENTOS (L4: swap+img2img intercalados y ajuste adaptativo; T4: swaps ya en caché).
#    Candid se aplica DESPUÉS de puntuar y solo al mejor: su viñeta/color cambian todo el fondo (falsearía FONDO_MAX) y es
#    determinista (no cambia el ranking entre intentos). Se guarda <stem>.png (con candid) y <stem>_sin_candid.png.
shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR + "/intentos")
REG, MEJOR = [], {}
for f, q in VALIDOS:
    stem = os.path.splitext(f)[0]; mejor = None; P, S = None, {}; print(f"\n=== {f} ===")
    for k in range(MAX_INTENTOS):
        P = siguiente_intento(k, P, S); t = time.time()
        try:
            sw = swap(q, P["blend"]); swap_bgr = cv2.imread(sw)
            torch.cuda.empty_cache(); fin, zona = realismo(swap_bgr, P["semilla"], P["strength"]); S = puntuar(fin, swap_bgr, zona)
            cv2.imwrite(f"{OUT_DIR}/intentos/{stem}_i{k+1}.png", fin)
        except Exception as e:
            S = dict(sim_swap=None, sim=None, cara=False, fondo=None, score=-9.0, pasa=False); fin = None; print(f"  ❌ intento {k+1}: {type(e).__name__}: {str(e)[:200]}"); torch.cuda.empty_cache()
        REG.append(dict(destino=f, intento=k+1, **P, **S, seg=round(time.time()-t, 1)))
        print(f"  intento {k+1}: sem {P['semilla']} str {P['strength']} blend {P['blend']} → ArcFace {S['sim']} (swap {S['sim_swap']}) cara {S['cara']} fondo {S['fondo']} · {'PASA' if S['pasa'] else 'no pasa'} · {time.time()-t:.0f}s")
        if fin is not None and (mejor is None or S["score"] > mejor[0]): mejor = (S["score"], k+1, fin)
        if S["pasa"]: break
    if mejor:
        MEJOR[f] = mejor[1]; print(f"  → mejor: intento {mejor[1]} (score {mejor[0]})")
        if APLICAR_CANDID:
            cv2.imwrite(f"{OUT_DIR}/{stem}_sin_candid.png", mejor[2])
            guardar(candid(mejor[2], [list(map(float, c.bbox)) for c in caras(mejor[2])]), f"{OUT_DIR}/{stem}")
        else: cv2.imwrite(f"{OUT_DIR}/{stem}.png", mejor[2])

In [ ]:
# 10) Reporte: tabla con puntuaciones por intento (★ = intento guardado) + CSV + miniaturas destino | mejor resultado
import pandas as pd, matplotlib.pyplot as plt
df = pd.DataFrame(REG); df["guardado"] = [("★" if MEJOR.get(r.destino) == r.intento else "") for r in df.itertuples()]
df.to_csv(f"{OUT_DIR}/reporte.csv", index=False); pd.set_option("display.width", 250); display(df)
print(f"MODO {MODO} · Z-Image {DT_ACT} · lado {A_SIDE} · LoRA {LORA_USADO} · candid {APLICAR_CANDID}")
print(f"Umbral ArcFace {UMBRAL_ARCFACE} · fondo máx {FONDO_MAX} · destinos que pasan: {sum(df[(df.destino==d)].pasa.any() for d in df.destino.unique())}/{df.destino.nunique()}")
hechos = [f for f, _ in VALIDOS if f in MEJOR]
fig, axs = plt.subplots(max(1, len(hechos)), 1, figsize=(10, 3.6*max(1, len(hechos))), squeeze=False)
h = lambda im: cv2.resize(im, (int(im.shape[1]*300/im.shape[0]), 300), interpolation=cv2.INTER_AREA)
for ax, f in zip(axs.ravel(), hechos):
    d = cv2.imread(os.path.join(PNG_DST, os.path.splitext(f)[0] + ".png")); r = next(cv2.imread(p) for p in (f"{OUT_DIR}/{os.path.splitext(f)[0]}.png", f"{OUT_DIR}/{os.path.splitext(f)[0]}.jpg") if os.path.exists(p))
    ax.imshow(cv2.cvtColor(np.hstack([h(d), h(r)]), cv2.COLOR_BGR2RGB)); ax.set_title(f"{f}: destino | mejor (intento {MEJOR[f]})", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig("/content/miniaturas.png", dpi=70); plt.show()

In [ ]:
# 11) Zip de resultados (mejores + intentos + reporte.csv) y descarga
shutil.make_archive("/content/resultados", "zip", OUT_DIR)
print("Zip:", "/content/resultados.zip", os.path.getsize("/content/resultados.zip")//1024, "KB")
try:
    from google.colab import files; files.download("/content/resultados.zip")
except Exception as e: print("files.download no disponible aquí:", type(e).__name__)